In [1]:
import numpy as np
import pandas as pd
from scipy import stats

In [2]:
active_validators_size = pd.read_csv('../int/active_validators_size_change.csv')
active_validators_category = pd.read_csv('../int/active_validators_category_change.csv')
active_validators_pool = pd.read_csv('../int/active_validators_pool_change.csv')

In [3]:
active_validators_pool = active_validators_pool.drop(columns=('Unnamed: 0'))
active_validators_category = active_validators_category.drop(columns=('Unnamed: 0'))
active_validators_size = active_validators_size.drop(columns=('Unnamed: 0'))

In [4]:
eth_price = pd.read_csv('../int/eth_price.csv')
eth_price = eth_price[eth_price['slot'] >= 6206400]
eth_price

,slot,price,price_pct_change
6174901,6206400,1892.938911,-0.008871
6174902,6206401,1892.938911,-0.008871
6174903,6206402,1892.938911,-0.008871
6174904,6206403,1892.938911,-0.008871
6174905,6206404,1892.938911,-0.008871
...,...,...,...
8954672,8986171,2976.090208,-0.014066
8954673,8986172,2976.090208,-0.014066
8954674,8986173,2976.090208,-0.014066
8954675,8986174,2976.090208,-0.014066


In [5]:

# Merge the two DataFrames on the slot column
price_elasticity_size = pd.merge(active_validators_size, eth_price[['slot', 'price_pct_change']], on='slot')

# Drop rows with infinite values
price_elasticity_size.replace([np.inf, -np.inf], np.nan, inplace=True)

# Calculate elasticity for 'total' first
price_elasticity_size['elasticity_total'] = price_elasticity_size['total'] / price_elasticity_size['price_pct_change']
price_elasticity_size['elasticity_total'] = price_elasticity_size['elasticity_total'].replace([np.inf, -np.inf], np.nan)

# Initialize dictionaries to store elasticity values, standard deviations, t-statistics, p-values, and number of valid rows (N)
elasticity = {}
standard_deviations = {}
t_statistics = {}
p_values = {}
valid_counts = {}

# Calculate elasticity for the 'total' column
valid_total_elasticity = price_elasticity_size['elasticity_total'].dropna()
elasticity['total'] = valid_total_elasticity.mean()
standard_deviations['total'] = valid_total_elasticity.std()
t_statistics['total'] = '-'
p_values['total'] = '-'
valid_counts['total'] = len(valid_total_elasticity)

# Calculate elasticity for each column except 'total'
columns = ['1', '2-5', '6-19', '20-99', '100+']
for col in columns:
    price_elasticity_size[f'elasticity_{col}'] = price_elasticity_size[col] / price_elasticity_size['price_pct_change']
    
    # Replace infinite values with NaN
    price_elasticity_size[f'elasticity_{col}'] = price_elasticity_size[f'elasticity_{col}'].replace([np.inf, -np.inf], np.nan)
    
    # Drop NaN values for calculation purposes
    valid_elasticity = price_elasticity_size[f'elasticity_{col}'].dropna()
    
    elasticity[col] = valid_elasticity.mean()
    standard_deviations[col] = valid_elasticity.std()
    
    # Perform a two-sample t-test against the 'total' elasticity
    t_stat, p_value = stats.ttest_ind(valid_elasticity, valid_total_elasticity, equal_var=False)
    t_statistics[col] = t_stat
    p_values[col] = p_value
    
    # Store the number of valid rows
    valid_counts[col] = len(valid_elasticity)

# Print the results in the requested format
print("Elasticity Analysis Results:")
print(f'total: Mean Elasticity = {elasticity["total"]}, t(Mean) = {t_statistics["total"]}, SD = {standard_deviations["total"]}, N = {valid_counts["total"]}, p-value = {p_values["total"]}')
for col in columns:
    print(f'{col}: Mean Elasticity = {elasticity[col]}, t(Mean) = {t_statistics[col]}, SD = {standard_deviations[col]}, N = {valid_counts[col]}, p-value = {p_values[col]}')

# Display the DataFrame with elasticity columns
print(price_elasticity_size)

Elasticity Analysis Results:
total: Mean Elasticity = -0.027549165749727895, t(Mean) = -, SD = 10.645470444967186, N = 9266, p-value = -
1: Mean Elasticity = -0.09491866780168519, t(Mean) = -0.22025956502564054, SD = 27.45058947310122, N = 9266, p-value = 0.825672759235415
2-5: Mean Elasticity = -0.15819833877361061, t(Mean) = -0.5626740584449123, SD = 19.652958328792536, N = 9266, p-value = 0.5736656907419627
6-19: Mean Elasticity = 0.14643802379302254, t(Mean) = 0.5621842082962709, SD = 27.824025632044574, N = 9266, p-value = 0.5740010901579706
20-99: Mean Elasticity = 0.28493976194960285, t(Mean) = 0.8536328085261117, SD = 33.5913923057921, N = 9266, p-value = 0.39332688236689584
100+: Mean Elasticity = -0.042685635772920456, t(Mean) = -0.09390140539209901, SD = 11.28898395357998, N = 9266, p-value = 0.9251884961672248
           slot         1      100+       2-5     20-99     6-19     total  \
0     6206400.0  0.015042 -0.015342  0.000000  0.018732  0.00000 -0.012792   
1     6206

In [6]:

# Merge the two DataFrames on the slot column
price_elasticity_category = pd.merge(active_validators_category, eth_price[['slot', 'price_pct_change']], on='slot')

# Drop rows with infinite values
price_elasticity_category.replace([np.inf, -np.inf], np.nan, inplace=True)

# Calculate elasticity for 'total' first
price_elasticity_category['elasticity_total'] = price_elasticity_category['total'] / price_elasticity_category['price_pct_change']
price_elasticity_category['elasticity_total'] = price_elasticity_category['elasticity_total'].replace([np.inf, -np.inf], np.nan)

# Initialize dictionaries to store elasticity values, standard deviations, t-statistics, p-values, and number of valid rows (N)
elasticity = {}
standard_deviations = {}
t_statistics = {}
p_values = {}
valid_counts = {}

# Calculate elasticity for the 'total' column
valid_total_elasticity = price_elasticity_category['elasticity_total'].dropna()
elasticity['total'] = valid_total_elasticity.mean()
standard_deviations['total'] = valid_total_elasticity.std()
t_statistics['total'] = '-'
p_values['total'] = '-'
valid_counts['total'] = len(valid_total_elasticity)

# Calculate elasticity for each column except 'total'
columns = active_validators_category.columns.drop('slot')
for col in columns:
    price_elasticity_category[f'elasticity_{col}'] = price_elasticity_category[col] / price_elasticity_category['price_pct_change']
    
    # Replace infinite values with NaN
    price_elasticity_category[f'elasticity_{col}'] = price_elasticity_category[f'elasticity_{col}'].replace([np.inf, -np.inf], np.nan)
    
    # Drop NaN values for calculation purposes
    valid_elasticity = price_elasticity_category[f'elasticity_{col}'].dropna()
    
    elasticity[col] = valid_elasticity.mean()
    standard_deviations[col] = valid_elasticity.std()
    
    # Perform a two-sample t-test against the 'total' elasticity
    t_stat, p_value = stats.ttest_ind(valid_elasticity, valid_total_elasticity, equal_var=False)
    t_statistics[col] = t_stat
    p_values[col] = p_value
    
    # Store the number of valid rows
    valid_counts[col] = len(valid_elasticity)

# Print the results in the requested format
print("Elasticity Analysis Results:")
print(f'total: Mean Elasticity = {elasticity["total"]}, t(Mean) = {t_statistics["total"]}, SD = {standard_deviations["total"]}, N = {valid_counts["total"]}, p-value = {p_values["total"]}')
for col in columns:
    print(f'{col}: Mean Elasticity = {elasticity[col]}, t(Mean) = {t_statistics[col]}, SD = {standard_deviations[col]}, N = {valid_counts[col]}, p-value = {p_values[col]}')

# Display the DataFrame with elasticity columns
print(price_elasticity_category)

Elasticity Analysis Results:
total: Mean Elasticity = -0.027549165749727895, t(Mean) = 0.0, SD = 10.645470444967186, N = 9266, p-value = 1.0
CEX: Mean Elasticity = 0.1432211773790807, t(Mean) = 0.7000274588334598, SD = 20.930831662504964, N = 9266, p-value = 0.48392199308387696
Liquid Restaking: Mean Elasticity = -6.2982155094831, t(Mean) = -1.8284218558159717, SD = 329.9571342732111, N = 9266, p-value = 0.06751833128052216
Liquid Staking: Mean Elasticity = -0.2413935459954165, t(Mean) = -1.155598750840481, SD = 14.282041886219044, N = 9266, p-value = 0.24786143539205024
Solo Stakers: Mean Elasticity = 0.32720455251882036, t(Mean) = 1.553694866698936, SD = 19.228862771099536, N = 9266, p-value = 0.12027910814803255
Staking Pools: Mean Elasticity = 0.06165566592325467, t(Mean) = 0.11861962324602415, SD = 71.6028756251507, N = 9266, p-value = 0.9055791807657818
Unidentified: Mean Elasticity = -0.24342917710423276, t(Mean) = -0.7442507326416652, SD = 25.812519144635157, N = 9266, p-value 

In [7]:

# Merge the two DataFrames on the slot column
price_elasticity_pool = pd.merge(active_validators_pool, eth_price[['slot', 'price_pct_change']], on='slot')

# Drop rows with infinite values
price_elasticity_pool.replace([np.inf, -np.inf], np.nan, inplace=True)

# Calculate elasticity for 'total' first
price_elasticity_pool['elasticity_total'] = price_elasticity_pool['total'] / price_elasticity_pool['price_pct_change']
price_elasticity_pool['elasticity_total'] = price_elasticity_pool['elasticity_total'].replace([np.inf, -np.inf], np.nan)

# Initialize dictionaries to store elasticity values, standard deviations, t-statistics, p-values, and number of valid rows (N)
elasticity = {}
standard_deviations = {}
t_statistics = {}
p_values = {}
valid_counts = {}

# Calculate elasticity for the 'total' column
valid_total_elasticity = price_elasticity_pool['elasticity_total'].dropna()
elasticity['total'] = valid_total_elasticity.mean()
standard_deviations['total'] = valid_total_elasticity.std()
t_statistics['total'] = '-'
p_values['total'] = '-'
valid_counts['total'] = len(valid_total_elasticity)

# Calculate elasticity for each column except 'total'
columns = ['Lido', 'Coinbase', 'Binance', 'Rocketpool', 'Kraken', 'OKX', 'Bitcoin Suisse', 'Ledger Live', 'Ether.Fi', 'Mantle', 'Other Stakers']
for col in columns:
    price_elasticity_pool[f'elasticity_{col}'] = price_elasticity_pool[col] / price_elasticity_pool['price_pct_change']
    
    # Replace infinite values with NaN
    price_elasticity_pool[f'elasticity_{col}'] = price_elasticity_pool[f'elasticity_{col}'].replace([np.inf, -np.inf], np.nan)
    
    # Drop NaN values for calculation purposes
    valid_elasticity = price_elasticity_pool[f'elasticity_{col}'].dropna()
    
    elasticity[col] = valid_elasticity.mean()
    standard_deviations[col] = valid_elasticity.std()
    
    # Perform a two-sample t-test against the 'total' elasticity
    t_stat, p_value = stats.ttest_ind(valid_elasticity, valid_total_elasticity, equal_var=False)
    t_statistics[col] = t_stat
    p_values[col] = p_value
    
    # Store the number of valid rows
    valid_counts[col] = len(valid_elasticity)

# Print the results in the requested format
print("Elasticity Analysis Results:")
print(f'total: Mean Elasticity = {elasticity["total"]}, t(Mean) = {t_statistics["total"]}, SD = {standard_deviations["total"]}, N = {valid_counts["total"]}, p-value = {p_values["total"]}')
for col in columns:
    print(f'{col}: Mean Elasticity = {elasticity[col]}, t(Mean) = {t_statistics[col]}, SD = {standard_deviations[col]}, N = {valid_counts[col]}, p-value = {p_values[col]}')

# Display the DataFrame with elasticity columns
print(price_elasticity_pool)

Elasticity Analysis Results:
total: Mean Elasticity = -0.027549165749727895, t(Mean) = -, SD = 10.645470444967186, N = 9266, p-value = -
Lido: Mean Elasticity = -0.16207782216914438, t(Mean) = -0.7021982194473083, SD = 15.058907333744742, N = 9266, p-value = 0.4825653573140902
Coinbase: Mean Elasticity = 0.08856854857804392, t(Mean) = 0.40331216792457714, SD = 25.588165588599406, N = 9266, p-value = 0.6867255571738856
Binance: Mean Elasticity = 0.643473870185888, t(Mean) = 1.4498746367372943, SD = 43.25998111251722, N = 9266, p-value = 0.14712368544597274
Rocketpool: Mean Elasticity = -0.37582408274826073, t(Mean) = -0.7234839261075986, SD = 45.09884360397234, N = 9266, p-value = 0.46939906457862635
Kraken: Mean Elasticity = -0.07439303734850668, t(Mean) = -0.15179705077540315, SD = 27.732391262238416, N = 9266, p-value = 0.8793495735063213
OKX: Mean Elasticity = -1.0009418004418844, t(Mean) = -2.5668636493983854, SD = 34.916477899444764, N = 9266, p-value = 0.010275433335730841
Bitcoi